<a href="https://colab.research.google.com/github/ad7x/RAG-using-cosine-similarity/blob/main/RAG_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Here I have Implemented RAG from screct with text document to answer question based to cosine similarity and embedding text into vector.

I have used gemini API to achieve this RAG Work.

Installing dependency for gemini API

In [23]:
!pip install -qU google-genai

In [24]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

Importing Gemini API to be used for embedding and retrieval


In [25]:
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

loading document and and splitting it

In [26]:
# Load your text document
file_path = 'doc.txt'

dataset = []
with open(file_path, 'r', encoding='utf-8') as file:
    # Read line by line or split by paragraphs/sentences depending on your text structure
    dataset = [line.strip() for line in file.readlines() if len(line.strip()) > 0]

print(f'Loaded {len(dataset)} entries from {file_path}')

Loaded 38 entries from doc.txt


using gemini-embedding-001 model for embedding.

In [34]:
EMBEDDING_MODEL = 'gemini-embedding-001'
VECTOR_DB = []

def add_chunk_to_database(chunk):
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=chunk
    )
    # Extract the float values from the response
    embedding = response.embeddings[0].values
    VECTOR_DB.append((chunk, embedding))

# Populate your vector database
for i, chunk in enumerate(dataset[:] ):
    add_chunk_to_database(chunk)
    print(f'Added chunk {i+1} to the database')

def cosine_similarity(a, b):
    dot_product = sum([x * y for x, y in zip(a, b)])
    norm_a = sum([x ** 2 for x in a]) ** 0.5
    norm_b = sum([x ** 2 for x in b]) ** 0.5
    return dot_product / (norm_a * norm_b)

def retrieve(query, top_n=3):
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=query
    )
    query_embedding = response.embeddings[0].values

    similarities = []
    for chunk, embedding in VECTOR_DB:
        similarity = cosine_similarity(query_embedding, embedding)
        similarities.append((chunk, similarity))

    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

Added chunk 1 to the database
Added chunk 2 to the database
Added chunk 3 to the database
Added chunk 4 to the database
Added chunk 5 to the database
Added chunk 6 to the database
Added chunk 7 to the database
Added chunk 8 to the database
Added chunk 9 to the database
Added chunk 10 to the database
Added chunk 11 to the database
Added chunk 12 to the database
Added chunk 13 to the database
Added chunk 14 to the database
Added chunk 15 to the database
Added chunk 16 to the database
Added chunk 17 to the database
Added chunk 18 to the database
Added chunk 19 to the database
Added chunk 20 to the database
Added chunk 21 to the database
Added chunk 22 to the database
Added chunk 23 to the database
Added chunk 24 to the database
Added chunk 25 to the database
Added chunk 26 to the database
Added chunk 27 to the database
Added chunk 28 to the database
Added chunk 29 to the database
Added chunk 30 to the database
Added chunk 31 to the database
Added chunk 32 to the database
Added chunk 33 to

In [32]:
# Ask a question related to your loaded document
input_query = "what is the color of china's flag?"
retrieved_knowledge = retrieve(input_query)

print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
    print(f' - (similarity: {similarity:.2f}) {chunk}\n')

# Construct the prompt context
context_text = '\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])
prompt = f"""You are a helpful chatbot.
Use only the following pieces of context to answer the question. Don't make up any new information:
{context_text}

Question: {input_query}
"""

# Generate response using Gemini
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=prompt,
)

print('Chatbot response:')
print(response.text)

Retrieved knowledge:
 - (similarity: 0.48) Description

 - (similarity: 0.47) An industrial robot manufactured by ABB

 - (similarity: 0.46) Electronics and electricals

Chatbot response:
Based on the context provided, there is no information available to answer what the color of China's flag is.
